In [ ]:
# Sanity check: environment
import sys
print(f"Python: {sys.version.split()[0]}")
try:
    import numpy as np
    print(f"NumPy: {np.__version__}")
except Exception as e:
    print("NumPy import failed:", e)

# Qallow QHIL - Colab Integration Demo

**Quantum Human-in-the-Loop** interactive optimization for Google Colab

This notebook demonstrates:
- Pure NumPy quantum circuit simulation
- Interactive human feedback integration
- Real-time quantum state visualization
- No local compilation required

🚀 Ready to run on Google Colab!

## 📦 Setup - Install Dependencies

Run this first in Colab to get required packages

In [ ]:
!pip install numpy matplotlib -q
print("✅ Dependencies installed!")

## 🔬 QHIL Core Implementation

Pure NumPy quantum simulator with human feedback integration

In [ ]:
import numpy as np
import json
from typing import Dict, List, Tuple
from dataclasses import dataclass, asdict
from datetime import datetime

class QuantumCircuit:
    """Simple quantum circuit simulator using NumPy"""
    def __init__(self, n_qubits: int):
        self.n_qubits = n_qubits
        self.gates = []
    
    def rx(self, qubit: int, angle: float):
        self.gates.append(('rx', qubit, angle))
    
    def rz(self, qubit: int, angle: float):
        self.gates.append(('rz', qubit, angle))
    
    def cnot(self, control: int, target: int):
        self.gates.append(('cnot', control, target))
    
    def simulate(self):
        """Simulate the circuit"""
        state = np.zeros(2**self.n_qubits, dtype=complex)
        state[0] = 1.0
        
        for gate_type, *params in self.gates:
            if gate_type == 'rx':
                qubit, angle = params
                state = self._apply_rx(state, qubit, angle)
            elif gate_type == 'rz':
                qubit, angle = params
                state = self._apply_rz(state, qubit, angle)
            elif gate_type == 'cnot':
                control, target = params
                state = self._apply_cnot(state, control, target)
        
        return state
    
    def _apply_rx(self, state, qubit, angle):
        """Apply Rx gate"""
        cos_half = np.cos(angle / 2)
        sin_half = np.sin(angle / 2)
        rx_matrix = np.array([[cos_half, -1j * sin_half],
                              [-1j * sin_half, cos_half]], dtype=complex)
        return self._apply_single_qubit_gate(state, qubit, rx_matrix)
    
    def _apply_rz(self, state, qubit, angle):
        """Apply Rz gate"""
        rz_matrix = np.array([[np.exp(-1j * angle / 2), 0],
                              [0, np.exp(1j * angle / 2)]], dtype=complex)
        return self._apply_single_qubit_gate(state, qubit, rz_matrix)
    
    def _apply_cnot(self, state, control, target):
        """Apply CNOT gate"""
        new_state = state.copy()
        for i in range(2**self.n_qubits):
            if (i >> control) & 1:
                j = i ^ (1 << target)
                new_state[i] = state[j]
                new_state[j] = state[i]
        return new_state
    
    def _apply_single_qubit_gate(self, state, qubit, gate_matrix):
        """Apply single qubit gate"""
        new_state = np.zeros_like(state)
        for i in range(2**self.n_qubits):
            qubit_val = (i >> qubit) & 1
            for out_val in [0, 1]:
                j = (i & ~(1 << qubit)) | (out_val << qubit)
                new_state[j] += gate_matrix[out_val, qubit_val] * state[i]
        return new_state

@dataclass
class QuantumState:
    """Represents a quantum state with metadata"""
    iteration: int
    depth: int
    fidelity: float
    entropy: float
    feedback: str
    params: List[float]
    timestamp: str

class QuantumHumanInTheLoopOptimizer:
    """Main QHIL optimization engine"""
    def __init__(self, n_qubits: int = 3, max_depth: int = 2):
        self.n_qubits = n_qubits
        self.max_depth = max_depth
        self.history = []
        self.current_params = np.random.randn(n_qubits * max_depth * 2) * 0.1
    
    def step(self, depth: int, params: np.ndarray):
        """Execute one optimization step"""
        circuit = QuantumCircuit(self.n_qubits)
        
        # Build circuit with parameters
        param_idx = 0
        for layer in range(depth):
            for q in range(self.n_qubits):
                if param_idx < len(params):
                    circuit.rx(q, params[param_idx])
                    param_idx += 1
                if param_idx < len(params):
                    circuit.rz(q, params[param_idx])
                    param_idx += 1
            
            # Entangling layer
            for q in range(self.n_qubits - 1):
                circuit.cnot(q, q + 1)
        
        state_vector = circuit.simulate()
        fidelity, entropy = self.compute_metrics(state_vector)
        
        return state_vector, fidelity, entropy
    
    def compute_metrics(self, state_vector):
        """Compute fidelity and entropy"""
        fidelity = np.abs(np.max(np.abs(state_vector))) ** 2
        probs = np.abs(state_vector) ** 2
        entropy = -np.sum(probs[probs > 1e-10] * np.log2(probs[probs > 1e-10]))
        return float(fidelity), float(entropy)
    
    def apply_human_feedback(self, feedback: str, params: np.ndarray) -> np.ndarray:
        """Apply human feedback to adjust parameters"""
        feedback_map = {
            'ENTANGLE STRONG': lambda p: p * 1.3,
            'ENTANGLE MEDIUM': lambda p: p * 1.1,
            'DEEPEN STRONG': lambda p: p * 1.5,
            'DEEPEN MEDIUM': lambda p: p * 1.2,
            'STABILIZE': lambda p: p * 0.95,
            'EXPLORE': lambda p: p + np.random.randn(len(p)) * 0.1
        }
        
        if feedback in feedback_map:
            return feedback_map[feedback](params)
        return params

print("✅ QHIL Core Implementation Loaded!")

## 🎯 Run Automated Demo

Execute 5 iterations with predefined feedback sequence

In [ ]:
# Initialize optimizer
optimizer = QuantumHumanInTheLoopOptimizer(n_qubits=3, max_depth=2)

# Predefined feedback sequence for demo
feedback_sequence = [
    "ENTANGLE STRONG",
    "DEEPEN MEDIUM", 
    "STABILIZE",
    "EXPLORE",
    "ENTANGLE MEDIUM"
]

print("🚀 Starting QHIL Demo (5 iterations)\n")
print("=" * 70)

results = []
params = optimizer.current_params

for i, feedback in enumerate(feedback_sequence, 1):
    # Execute quantum circuit
    state, fidelity, entropy = optimizer.step(depth=2, params=params)
    
    # Record state
    quantum_state = QuantumState(
        iteration=i,
        depth=2,
        fidelity=fidelity,
        entropy=entropy,
        feedback=feedback,
        params=params.tolist(),
        timestamp=datetime.now().isoformat()
    )
    
    results.append(quantum_state)
    
    # Display progress
    print(f"Iteration {i}:")
    print(f"  Fidelity: {fidelity:.6f}")
    print(f"  Entropy:  {entropy:.6f}")
    print(f"  Feedback: {feedback}")
    print()
    
    # Apply feedback for next iteration
    params = optimizer.apply_human_feedback(feedback, params)

print("=" * 70)
print("✅ Demo Complete!")
print(f"📊 Total iterations: {len(results)}")
print(f"📈 Final fidelity: {results[-1].fidelity:.6f}")
print(f"📉 Final entropy: {results[-1].entropy:.6f}")

## 📊 Visualize Results

Plot fidelity and entropy evolution over iterations

In [ ]:
import matplotlib.pyplot as plt

iterations = [r.iteration for r in results]
fidelities = [r.fidelity for r in results]
entropies = [r.entropy for r in results]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Fidelity plot
ax1.plot(iterations, fidelities, 'b-o', linewidth=2, markersize=8)
ax1.set_xlabel('Iteration', fontsize=12)
ax1.set_ylabel('Fidelity', fontsize=12)
ax1.set_title('Quantum State Fidelity Evolution', fontsize=14, fontweight='bold')
ax1.grid(True, alpha=0.3)
ax1.set_ylim([0, 1])

# Entropy plot
ax2.plot(iterations, entropies, 'r-s', linewidth=2, markersize=8)
ax2.set_xlabel('Iteration', fontsize=12)
ax2.set_ylabel('Entropy', fontsize=12)
ax2.set_title('Quantum State Entropy Evolution', fontsize=14, fontweight='bold')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("📈 Visualization complete!")

## 💾 Export Results

Save results as JSON for further analysis

In [ ]:
# Convert results to JSON
results_json = {
    "experiment": "QHIL Colab Demo",
    "n_qubits": optimizer.n_qubits,
    "max_depth": optimizer.max_depth,
    "total_iterations": len(results),
    "results": [asdict(r) for r in results]
}

# Save to file
output_file = "qhil_results.json"
with open(output_file, 'w') as f:
    json.dump(results_json, f, indent=2)

print(f"✅ Results saved to: {output_file}")
print(f"📊 Total data points: {len(results)}")

# Display summary
print("\n" + "=" * 70)
print("SUMMARY")
print("=" * 70)
print(f"Fidelity improvement: {results[-1].fidelity - results[0].fidelity:+.6f}")
print(f"Entropy change:       {results[-1].entropy - results[0].entropy:+.6f}")
print(f"Best fidelity:        {max(fidelities):.6f} (iteration {fidelities.index(max(fidelities))+1})")
print(f"Lowest entropy:       {min(entropies):.6f} (iteration {entropies.index(min(entropies))+1})")

## 🎮 Interactive Mode (Optional)

Run this cell for manual feedback input

In [ ]:
# Interactive QHIL session
print("🎮 Interactive QHIL Mode")
print("=" * 70)
print("\nAvailable feedback commands:")
print("  - ENTANGLE STRONG / MEDIUM")
print("  - DEEPEN STRONG / MEDIUM")
print("  - STABILIZE")
print("  - EXPLORE")
print("  - quit (to exit)")
print()

interactive_optimizer = QuantumHumanInTheLoopOptimizer(n_qubits=3, max_depth=2)
params = interactive_optimizer.current_params
iteration = 0

# Example: Run one iteration (uncomment for interactive use in Colab)
# while True:
#     iteration += 1
#     state, fidelity, entropy = interactive_optimizer.step(depth=2, params=params)
#     
#     print(f"\n📊 Iteration {iteration}:")
#     print(f"   Fidelity: {fidelity:.6f}")
#     print(f"   Entropy:  {entropy:.6f}")
#     
#     feedback = input("\n💬 Your feedback: ").strip().upper()
#     
#     if feedback == "QUIT":
#         print("👋 Session ended!")
#         break
#     
#     params = interactive_optimizer.apply_human_feedback(feedback, params)

print("✅ Interactive mode ready (uncomment code block to enable)")

---

## 🔗 Next Steps

**Local Development:**
```bash
git clone https://github.com/xingxerx/Qallow.git
cd Qallow
./bootstrap.sh
source .venv/bin/activate
python3 quantum_human_loop.py
```

**Full Documentation:**
- [QHIL Documentation](https://github.com/xingxerx/Qallow/blob/main/QHIL_DOCUMENTATION.md)
- [QML Integration Guide](https://github.com/xingxerx/Qallow/blob/main/QML_INDEX.md)
- [Complete README](https://github.com/xingxerx/Qallow/blob/main/README.md)

**Features Not Available in Colab:**
- Native C/CUDA quantum simulator
- Full 13-phase execution pipeline
- Hardware-accelerated optimization
- REST API server mode

🚀 **This notebook provides the core QHIL algorithm - perfect for research, education, and rapid experimentation!**